# Notebook 37 - Audit follow-ups: twenty seeds, the 11.8% student, natural-prior validation

Three cheap experiments the hostile read asked for, with readings fixed in stage 2.

**Twenty seeds on the two decisive cells.** Five methods by twenty recovery seeds under minimal recovery on both architectures, every model recalibrated on the NB31 slice, analysed under both normalisations. S1: the paired-contrast minimum detectable difference on recalibrated AWBIR is at most 0.03 in both cells. S2 (descriptive): which method effects are detectable, their sizes with intervals after Holm correction over the ten pairs, whether the shallow ordering holds, and the same contrasts on ground-truth HSR and benign escalation. The five original seeds are a subset, so the five-seed result can be compared directly.

**The frozen random 40% student.** Hash-verified, evaluated on validation at the class-balanced subsample and at the natural prior, under stored statistics, batch statistics and after recalibration, with its fourteen siblings and the teacher; for its benign rows, where escalations go and whether escalated rows are more extreme than retained ones. S3 reading rule: natural-prior validation escalation at or above 5% means the test-time 11.8% is a prior or subsample effect visible on validation; below 5% it is a validation-versus-test discrepancy reported as such. The test partition is not touched.

**Natural-prior validation.** All frozen students and both teachers on the full validation partition. S4: a teacher's validation-to-test drop is a prior effect if natural-prior validation is within 0.03 of its test value.

**Stages.** 1 bootstrap, 2 pre-registration, 3 helpers with both evaluation priors, 4 twenty-seed runs (resumable per cell), 5 analysis, 6 the frozen student and siblings, 7 natural-prior validation, 8 verdict and figures. GPU recommended; stage 4 is 200 one-epoch recoveries.

In [ ]:
# Stage 1 - bootstrap
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, sys, json, copy, hashlib, itertools
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn, yaml
import matplotlib.pyplot as plt

REPO = Path("/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression")
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from src.saber.bridge_ciciot import load_bridge
from src.saber.taxonomy import ciciot2023_taxonomy, DEFAULT_COST_PROFILES
from src.saber.metrics import full_model_audit, action_weighted_boundary_inversion_rate
from src.saber.surgery import prune_cnn1d_channels

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
R = REPO / "results/saber"
OUT = R / "37_audit_followups"; OUT.mkdir(parents=True, exist_ok=True)
MODEL_DIR = REPO / "models/ciciot2023"
print("device:", DEVICE)


In [ ]:
# Stage 2 - pre-registration
PREREG = {
    "arm": "S_audit_followups",
    "S_twenty_seeds": {
        "design": ("5 methods x 20 recovery seeds on the frozen 40% structures, minimal recovery (1 unit on a seeded 10% subset), "
                   "both architectures; every final model recalibrated on the fixed slice of NB31 (seed 2026); analyses under both "
                   "normalisations with recalibrated primary; the 5 original seeds are included so the 5-seed result is a subset"),
        "S1": "with 20 seeds, the paired-contrast minimum detectable difference (half-width of the 95% interval) on recalibrated AWBIR is at most 0.03 in both cells",
        "S2": ("descriptive: which method effects are detectable at 20 seeds (permutation, and paired contrasts with Holm correction over the "
               "10 pairs per cell), their sizes with intervals, and whether the shallow ordering of Section 7.12 (V-C lowest, magnitude highest) holds"),
        "S2_ground_truth": "the same contrasts on ground-truth HSR (balanced profile) and benign escalation, so the effect is stated in a non-teacher-referenced endpoint"},
    "S_frozen_random40": {
        "design": ("the frozen shallow random 40% student (17b registry, hash-verified) evaluated on validation at the class-balanced "
                   "subsample and at the natural prior (all validation rows), in eval mode, under batch statistics, and after recalibration; "
                   "the same for its 14 frozen siblings and the teacher; for benign rows, the predicted-class distribution of escalations and "
                   "the feature extremity (max |z|) of escalated versus retained benign rows"),
        "S3": ("reading rule fixed in advance: if its natural-prior validation benign escalation is at least 0.05, the test-time 11.8% is a "
               "prior/subsample effect visible on validation; if it stays below 0.05, it is a validation-versus-test discrepancy the paper "
               "reports as such without touching test")},
    "S_natural_prior": {
        "design": "all 25 frozen students and both teachers evaluated on the full validation partition at its natural class prior, same code path as the balanced subsample",
        "S4": ("the validation-to-test difference in fine macro-F1 for each teacher (0.51 to 0.40 deep, 0.56 to 0.53 shallow) is a prior effect if "
               "natural-prior validation is within 0.03 of the test value; otherwise it is reported as unexplained")},
    "no_test_access": True, "no_new_selection": True,
}
(OUT / "S_PREREGISTRATION.json").write_text(json.dumps(PREREG, indent=2)); print(json.dumps(PREREG, indent=2))
METHODS = ["random", "magnitude", "taylor", "fisher", "saber_v2"]
SEEDS20 = [101, 211, 307, 401, 503, 613, 719, 823, 907, 1013, 1109, 1201, 1303, 1409, 1511, 1607, 1709, 1801, 1907, 2003]
SUBSET_FRACTION = 0.10; CAL_SEED = 2026; COLLAPSE_B2A = 0.10


In [ ]:
# Stage 3 - data, teachers, helpers (both evaluation priors)
TRAIN_LOADER, VAL_LOADER, _T, SHALLOW_TEACHER, CLASS_NAMES = load_bridge()
taxonomy = ciciot2023_taxonomy(CLASS_NAMES); robust_graph = pd.read_csv(R / "14_risk_graph/asvg_edges_robust.csv")
N_CLASSES = len(CLASS_NAMES); SABER_CFG = yaml.safe_load(open(REPO / "config/saber.yaml"))
MIN_W = {"shallow": int(SABER_CFG["groups"]["minimum_remaining_per_layer"]), "deep": 8}
FAM = np.asarray(taxonomy.class_to_family_index); FAMILIES = list(taxonomy.families); BENIGN = int(taxonomy.benign_index)


class DeepCNN1D(nn.Module):
    def __init__(self, n_classes=34):
        super().__init__()
        def blk(i, o):
            return [nn.Conv1d(i, o, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(o)]
        self.conv = nn.Sequential(*blk(1, 64), *blk(64, 128), nn.MaxPool1d(2), *blk(128, 128), *blk(128, 256))
        self.pool = nn.AdaptiveAvgPool1d(1); self.head = nn.Linear(256, n_classes)
    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        return self.head(self.pool(self.conv(x.float())).squeeze(-1))


TEACHERS = {"shallow": SHALLOW_TEACHER.to(DEVICE).eval()}
_dt = DeepCNN1D(N_CLASSES); _dt.load_state_dict(torch.load(MODEL_DIR / "deepcnn1d_g5_seed0.pt", map_location="cpu", weights_only=False)["state_dict"])
TEACHERS["deep"] = _dt.to(DEVICE).eval()

Xv, Yv = VAL_LOADER.dataset.tensors; VAL_Y_ALL = Yv.numpy()
_rng = np.random.default_rng(0)
_idx = np.concatenate([_rng.permutation(np.where(VAL_Y_ALL == c)[0])[:4000] for c in range(N_CLASSES) if (VAL_Y_ALL == c).sum() > 0])
_idx = np.random.default_rng(12345).permutation(_idx)
_nat = np.random.default_rng(12345).permutation(len(VAL_Y_ALL))          # fixed shuffle: chunked probes must not see class-grouped rows
PRIORS = {"balanced": (Xv[_idx].to(DEVICE), VAL_Y_ALL[_idx]), "natural": (Xv[_nat].to(DEVICE), VAL_Y_ALL[_nat])}
EXAMPLE_INPUT = Xv[:8].float().to(DEVICE)
Xt, Yt = TRAIN_LOADER.dataset.tensors; N_TRAIN = len(Xt)
_counts = np.bincount(Yt.numpy(), minlength=N_CLASSES); _w = np.zeros_like(_counts, dtype=np.float64)
_w[_counts > 0] = 1.0 / np.sqrt(_counts[_counts > 0]); _w[_counts > 0] /= _w[_counts > 0].mean()
CLASS_W = torch.tensor(_w, dtype=torch.float32, device=DEVICE)
_cal_idx = torch.randperm(N_TRAIN, generator=torch.Generator().manual_seed(CAL_SEED))[:50 * 1024]
CAL_LOADER = torch.utils.data.DataLoader(torch.utils.data.Subset(TRAIN_LOADER.dataset, _cal_idx.tolist()), batch_size=1024, shuffle=False)
print("balanced rows:", len(_idx), "| natural rows:", len(VAL_Y_ALL), "| benign rows natural:", int((VAL_Y_ALL == BENIGN).sum()))


def forward_logits(model, X):
    model.eval()
    with torch.no_grad():
        return torch.cat([model(X[i:i + 8192]).cpu() for i in range(0, len(X), 8192)]).numpy()


def forward_logits_batch_stats(model, X):
    saved = {n: (m.running_mean.clone(), m.running_var.clone(), m.momentum, m.num_batches_tracked.clone())
             for n, m in model.named_modules() if isinstance(m, nn.BatchNorm1d) and m.running_mean is not None}
    model.train()
    for n, m in model.named_modules():
        if n in saved: m.momentum = 0.0
    with torch.no_grad():
        out = torch.cat([model(X[i:i + 8192]).cpu() for i in range(0, len(X), 8192)]).numpy()
    for n, m in model.named_modules():
        if n in saved:
            rm, rv, mom, nbt = saved[n]; m.running_mean.copy_(rm); m.running_var.copy_(rv); m.momentum = mom; m.num_batches_tracked.copy_(nbt)
    model.eval()
    return out


T_LOGITS = {(a, p): forward_logits(m, PRIORS[p][0]) for a, m in TEACHERS.items() for p in PRIORS}


def audit_logits(lg, arch, prior):
    X, Y = PRIORS[prior]
    a = full_model_audit(lg, Y, taxonomy, DEFAULT_COST_PROFILES)
    aw, _ = action_weighted_boundary_inversion_rate(T_LOGITS[(arch, prior)], lg, Y, robust_graph)
    return {"b2a": float(a["benign_to_attack_rate"]), "a2b": float(a["attack_to_benign_rate"]), "family_f1": float(a["family_macro_f1"]),
            "fine_f1": float(a["fine_macro_f1"]), "awbir": float(aw), "ece": float(a["ece15"]), "hsr_balanced": float(a["hsr_balanced_soc"]),
            "hsr_miss": float(a["hsr_miss_sensitive"]), "hsr_fatigue": float(a["hsr_alert_fatigue"])}


def audit(model, arch, prior="balanced", mode="eval"):
    X = PRIORS[prior][0]
    return audit_logits(forward_logits(model, X) if mode == "eval" else forward_logits_batch_stats(model, X), arch, prior)


def set_bn_momentum(model, momentum):
    for m in model.modules():
        if isinstance(m, nn.BatchNorm1d): m.momentum = momentum


def recalibrate_bn(model, n_batches=50):
    for m in model.modules():
        if isinstance(m, nn.BatchNorm1d): m.reset_running_stats(); m.momentum = None
    model.train()
    with torch.no_grad():
        for i, (xb, _) in enumerate(CAL_LOADER):
            if i >= n_batches: break
            model(xb.to(DEVICE))
    set_bn_momentum(model, 0.1); model.eval(); return model


def removed_path(arch, method, budget=0.40):
    return (R / f"17b_calibrated_checkpoint_freeze/{method}_r{int(round(budget * 100))}cal_removed_groups.csv") if arch == "shallow" \
        else (R / f"20b_depth_checkpoint_freeze/{method}_minimal_r40_removed_groups.csv")


def raw_student(arch, method, budget=0.40):
    rm = pd.read_csv(removed_path(arch, method, budget)); pm = {str(l): sorted(g["channel_index"].astype(int).tolist()) for l, g in rm.groupby("module_path")}
    st, _ = prune_cnn1d_channels(TEACHERS[arch], pm, EXAMPLE_INPUT, minimum_remaining_per_layer=MIN_W[arch]); return st.to(DEVICE)


def make_subset_loader(seed):
    sub = torch.randperm(N_TRAIN, generator=torch.Generator().manual_seed(seed))[: int(N_TRAIN * SUBSET_FRACTION)]
    return torch.utils.data.DataLoader(torch.utils.data.Subset(TRAIN_LOADER.dataset, sub.tolist()), batch_size=1024, shuffle=True, generator=torch.Generator().manual_seed(seed))


for _a, _m in TEACHERS.items():
    _X = PRIORS["balanced"][0]; _e1 = forward_logits(_m, _X); _p = forward_logits_batch_stats(_m, _X); _e2 = forward_logits(_m, _X)
    assert np.allclose(_e1, _e2, atol=1e-5) and not _m.training and not np.allclose(_e1, _p, atol=1e-6)
print("helpers verified")


In [ ]:
# Stage 4 - twenty seeds on the two decisive cells (minimal recovery), both normalisations
RUNS = OUT / "twenty_seed_runs.csv"
rows = pd.read_csv(RUNS).to_dict("records") if RUNS.exists() else []
done = {(r["architecture"], r["method"], r["seed"]) for r in rows}
print("complete:", len(done), "of", 2 * len(METHODS) * len(SEEDS20))
for arch in ["shallow", "deep"]:
    for method in METHODS:
        for seed in SEEDS20:
            if (arch, method, seed) in done:
                continue
            torch.manual_seed(seed); np.random.seed(seed)
            student = raw_student(arch, method); raw = audit(student, arch)
            loader = make_subset_loader(seed); opt = torch.optim.Adam(student.parameters(), lr=1e-3); lossf = nn.CrossEntropyLoss(weight=CLASS_W)
            student.train()
            for xb, yb in loader:
                opt.zero_grad(); lossf(student(xb.to(DEVICE)), yb.to(DEVICE)).backward(); opt.step()
            student.eval()
            tr, bs = audit(student, arch, "balanced", "eval"), audit(student, arch, "balanced", "batch")
            recalibrate_bn(student); rc = audit(student, arch, "balanced", "eval")
            row = {"architecture": arch, "method": method, "seed": seed, "raw_awbir": raw["awbir"], "is_collapse": bool(tr["b2a"] > COLLAPSE_B2A and bs["b2a"] <= COLLAPSE_B2A)}
            row.update({f"trained_{k}": v for k, v in tr.items()}); row.update({f"recal_{k}": v for k, v in rc.items()})
            rows.append(row); pd.DataFrame(rows).to_csv(RUNS, index=False)
            print(f"{arch} {method:9s} s{seed:4d}: awbir trained={tr['awbir']:.4f} recal={rc['awbir']:.4f} | hsr recal={rc['hsr_balanced']:.4f} | b2a recal={rc['b2a']:.4f}{'  COLLAPSE' if row['is_collapse'] else ''}")
runs = pd.DataFrame(rows); print("rows:", len(runs))


In [ ]:
# Stage 5 - analysis of the twenty-seed cells (S1, S2)
runs = pd.read_csv(OUT / "twenty_seed_runs.csv"); rng = np.random.default_rng(0)


def shares(x):
    grand = x.mean(); ss_tot = ((x - grand) ** 2).sum()
    ss_m = x.shape[0] * ((x.mean(axis=0) - grand) ** 2).sum(); ss_s = x.shape[1] * ((x.mean(axis=1) - grand) ** 2).sum()
    return float(ss_m / ss_tot), float(ss_s / ss_tot)


def perm_p(x, B=10000):
    m_obs, s_obs = shares(x); s_null, m_null = [], []
    for _ in range(B):
        xs = x.copy()
        for j in range(xs.shape[1]): xs[:, j] = rng.permutation(xs[:, j])
        s_null.append(shares(xs)[1])
        xm = x.copy()
        for i in range(xm.shape[0]): xm[i, :] = rng.permutation(xm[i, :])
        m_null.append(shares(xm)[0])
    return float(np.mean(np.array(m_null) >= m_obs)), float(np.mean(np.array(s_null) >= s_obs))


from scipy import stats
def contrasts(piv):
    out = []; n = len(piv); tcrit = stats.t.ppf(0.975, n - 1)
    for a, b in itertools.combinations(METHODS, 2):
        d = (piv[a] - piv[b]).values; m = d.mean(); se = d.std(ddof=1) / np.sqrt(n)
        p = float(stats.ttest_rel(piv[a], piv[b]).pvalue)
        out.append({"contrast": f"{a}-{b}", "mean": float(m), "ci_lo": float(m - tcrit * se), "ci_hi": float(m + tcrit * se), "half_width": float(tcrit * se), "p": p})
    # Holm correction over the 10 pairs
    ps = sorted([(o["p"], i) for i, o in enumerate(out)]); m_ = len(out); adj = {}
    running = 0.0
    for k, (p, i) in enumerate(ps):
        running = max(running, min(1.0, p * (m_ - k))); adj[i] = running
    for i, o in enumerate(out): o["p_holm"] = adj[i]; o["significant_holm"] = bool(adj[i] < 0.05)
    return out


analysis, rows_c = {}, []
for arch in ["shallow", "deep"]:
    sub = runs[runs.architecture == arch]; cell = {"n_seeds": int(sub.seed.nunique()), "collapses": int(sub.is_collapse.sum())}
    raw_spread = float(sub.groupby("method")["raw_awbir"].first().agg(lambda v: v.max() - v.min()))
    for norm in ["trained", "recal"]:
        for metric in ["awbir", "hsr_balanced", "b2a", "family_f1"]:
            piv = sub.pivot_table(index="seed", columns="method", values=f"{norm}_{metric}")[METHODS]
            p_m, p_s = perm_p(piv.values); cs = contrasts(piv)
            for c in cs: rows_c.append({"architecture": arch, "normalisation": norm, "metric": metric, **c})
            key = f"{norm}_{metric}"
            cell[key] = {"p_method": p_m, "p_seed": p_s, "means": {m: float(v) for m, v in piv.mean().items()},
                         "mde": float(np.median([c["half_width"] for c in cs])), "max_mde": float(max(c["half_width"] for c in cs)),
                         "n_holm_significant": int(sum(c["significant_holm"] for c in cs)), "max_abs_mean": float(max(abs(c["mean"]) for c in cs)),
                         "rank_rho": float(np.nanmean([piv.rank(axis=1).loc[a].corr(piv.rank(axis=1).loc[b], method="spearman") for a, b in itertools.combinations(piv.index, 2)]))}
            if metric == "awbir": cell[key]["rei"] = float(1 - (piv.max(axis=1) - piv.min(axis=1)).mean() / raw_spread) if raw_spread > 0 else float("nan")
    # 5-seed subset for comparison
    sub5 = sub[sub.seed.isin([101, 211, 307, 401, 503])]
    piv5 = sub5.pivot_table(index="seed", columns="method", values="recal_awbir")[METHODS]
    cell["five_seed_subset_recal_awbir"] = {"p_method": perm_p(piv5.values)[0], "mde": float(np.median([c["half_width"] for c in contrasts(piv5)]))}
    analysis[arch] = cell
    a = cell["recal_awbir"]
    print(f"{arch}: recal AWBIR means {dict((m, round(v, 4)) for m, v in a['means'].items())} | p_method {a['p_method']:.4f} p_seed {a['p_seed']:.4f} | "
          f"MDE median {a['mde']:.4f} max {a['max_mde']:.4f} | Holm-significant pairs {a['n_holm_significant']}/10 | REI {a['rei']:.3f} | rank rho {a['rank_rho']:.2f} | collapses {cell['collapses']}")
    h = cell["recal_hsr_balanced"]
    print(f"   recal HSR means {dict((m, round(v, 4)) for m, v in h['means'].items())} | p_method {h['p_method']:.4f} | Holm-significant {h['n_holm_significant']}/10 | max |diff| {h['max_abs_mean']:.4f}")
pd.DataFrame(rows_c).to_csv(OUT / "twenty_seed_contrasts.csv", index=False)
json.dump(analysis, open(OUT / "twenty_seed_analysis.json", "w"), indent=2)
S1 = all(analysis[a]["recal_awbir"]["max_mde"] <= 0.03 for a in analysis)
print("\nS1 (max paired MDE <= 0.03 AWBIR in both cells at 20 seeds):", S1)


In [ ]:
# Stage 6 - the frozen shallow random 40% student and its siblings, both priors, three normalisation states (S3)
reg = pd.read_csv(R / "17b_calibrated_checkpoint_freeze/shallow_frozen_model_registry.csv")
rows, benign_rows = [], []
for r in reg.itertuples():
    ck = REPO / str(r.checkpoint)
    assert hashlib.sha256(ck.read_bytes()).hexdigest() == r.checkpoint_sha256, f"hash mismatch {ck}"
    st = raw_student("shallow", r.method, float(r.target_flops)); st.load_state_dict(torch.load(ck, map_location="cpu", weights_only=False)["state_dict"]); st = st.to(DEVICE).eval()
    for prior in PRIORS:
        ev, bs = audit(st, "shallow", prior, "eval"), audit(st, "shallow", prior, "batch")
        st2 = copy.deepcopy(st); recalibrate_bn(st2); rc = audit(st2, "shallow", prior, "eval")
        rows.append({"method": r.method, "target_flops": float(r.target_flops), "prior": prior,
                     **{f"eval_{k}": v for k, v in ev.items()}, **{f"batch_{k}": v for k, v in bs.items()}, **{f"recal_{k}": v for k, v in rc.items()}})
        if r.method == "random" and np.isclose(r.target_flops, 0.40):
            X, Y = PRIORS[prior]; lg = forward_logits(st, X); pred = lg.argmax(1); ben = np.where(Y == BENIGN)[0]
            esc = ben[pred[ben] != BENIGN]; kept = ben[pred[ben] == BENIGN]
            absmax = X.abs().max(dim=1).values.cpu().numpy()
            dest = pd.Series([CLASS_NAMES[c] for c in pred[esc]]).value_counts().head(6).to_dict()
            benign_rows.append({"prior": prior, "benign_rows": int(len(ben)), "escalated": int(len(esc)), "rate": float(len(esc) / len(ben)),
                                "median_max_abs_z_escalated": float(np.median(absmax[esc])) if len(esc) else float("nan"),
                                "median_max_abs_z_retained": float(np.median(absmax[kept])) if len(kept) else float("nan"),
                                "share_escalated_with_abs_z_over_10": float((absmax[esc] > 10).mean()) if len(esc) else float("nan"),
                                "share_retained_with_abs_z_over_10": float((absmax[kept] > 10).mean()) if len(kept) else float("nan"),
                                "top_destinations": dest})
    print(f"{r.method:9s} {float(r.target_flops):.2f}: balanced b2a eval {rows[-2]['eval_b2a']:.4f} / natural b2a eval {rows[-1]['eval_b2a']:.4f} (batch {rows[-1]['batch_b2a']:.4f}, recal {rows[-1]['recal_b2a']:.4f})")
fro = pd.DataFrame(rows); fro.to_csv(OUT / "frozen_shallow_both_priors.csv", index=False)
ben = pd.DataFrame(benign_rows); ben.to_csv(OUT / "frozen_random40_benign_breakdown.csv", index=False)
print("\nrandom 40% benign breakdown:"); print(ben.drop(columns=["top_destinations"]).round(4).to_string(index=False)); print(ben.top_destinations.tolist())
r40 = fro[(fro.method == "random") & np.isclose(fro.target_flops, 0.40) & (fro.prior == "natural")].iloc[0]
S3 = "prior_or_subsample_effect_visible_on_validation" if r40.eval_b2a >= 0.05 else "validation_vs_test_discrepancy"
print("\nS3 reading:", S3, "| natural-prior validation b2a:", round(float(r40.eval_b2a), 4), "| test value in the locked audit: 0.118")


In [ ]:
# Stage 7 - natural-prior validation for all 25 frozen students and both teachers (S4)
test = pd.read_csv(R / "21_one_shot_test/test_model_summary.csv")
reg20 = pd.read_csv(R / "20b_depth_checkpoint_freeze/deep_frozen_model_registry.csv") if (R / "20b_depth_checkpoint_freeze/deep_frozen_model_registry.csv").exists() else None
rows = []
for arch, m in TEACHERS.items():
    for prior in PRIORS:
        a = audit(m, arch, prior); rows.append({"architecture": arch, "model": "dense", "variant": "teacher", "target_flops": np.nan, "prior": prior, **a})
fro = pd.read_csv(OUT / "frozen_shallow_both_priors.csv")
for r in fro.itertuples():
    rows.append({"architecture": "shallow", "model": r.method, "variant": "full recovery", "target_flops": r.target_flops, "prior": r.prior,
                 **{k: getattr(r, f"eval_{k}") for k in ["b2a", "a2b", "family_f1", "fine_f1", "awbir", "ece", "hsr_balanced", "hsr_miss", "hsr_fatigue"]}})
if reg20 is not None:
    for r in reg20.itertuples():
        ck = REPO / str(r.checkpoint)
        if not ck.exists(): print("missing deep checkpoint", ck); continue
        st = raw_student("deep", r.method); st.load_state_dict(torch.load(ck, map_location="cpu", weights_only=False)["state_dict"]); st = st.to(DEVICE).eval()
        for prior in PRIORS:
            a = audit(st, "deep", prior); rows.append({"architecture": "deep", "model": r.method, "variant": str(getattr(r, "variant", getattr(r, "regime", ""))), "target_flops": 0.40, "prior": prior, **a})
else:
    print("deep registry not found at the expected path; deep students skipped in S4 (teachers still evaluated)")
nat = pd.DataFrame(rows); nat.to_csv(OUT / "natural_prior_validation.csv", index=False)
# teachers: validation natural vs test
tv = {a: float(nat[(nat.architecture == a) & (nat.model == "dense") & (nat.prior == "natural")].fine_f1.iloc[0]) for a in TEACHERS}
_teach = test[(test.get("variant", pd.Series([""] * len(test))).astype(str) == "teacher") | (test.method.astype(str) == "dense")]
tt = {a: float(_teach[_teach.architecture == a].fine_macro_f1.iloc[0]) for a in TEACHERS}
S4 = {a: {"val_balanced_fine_f1": float(nat[(nat.architecture == a) & (nat.model == "dense") & (nat.prior == "balanced")].fine_f1.iloc[0]),
          "val_natural_fine_f1": tv[a], "test_fine_f1": tt[a], "within_0.03": bool(abs(tv[a] - tt[a]) <= 0.03)} for a in TEACHERS}
print(json.dumps(S4, indent=2))
tb = {a: {"val_natural_b2a": float(nat[(nat.architecture == a) & (nat.model == "dense") & (nat.prior == "natural")].b2a.iloc[0]),
          "test_b2a": float(_teach[_teach.architecture == a].benign_to_attack_rate.iloc[0])} for a in TEACHERS}
print("teacher benign escalation, natural-prior validation vs test:", tb)


In [ ]:
# Stage 8 - verdict and figures
analysis = json.load(open(OUT / "twenty_seed_analysis.json")); ben = pd.read_csv(OUT / "frozen_random40_benign_breakdown.csv")
fro = pd.read_csv(OUT / "frozen_shallow_both_priors.csv"); nat = pd.read_csv(OUT / "natural_prior_validation.csv"); test = pd.read_csv(R / "21_one_shot_test/test_model_summary.csv")
r40n = fro[(fro.method == "random") & np.isclose(fro.target_flops, 0.40) & (fro.prior == "natural")].iloc[0]
S1 = all(analysis[a]["recal_awbir"]["max_mde"] <= 0.03 for a in analysis)
S3 = "prior_or_subsample_effect_visible_on_validation" if r40n.eval_b2a >= 0.05 else "validation_vs_test_discrepancy"
_teach = test[(test.get("variant", pd.Series([""] * len(test))).astype(str) == "teacher") | (test.method.astype(str) == "dense")]
S4 = {a: bool(abs(float(nat[(nat.architecture == a) & (nat.model == "dense") & (nat.prior == "natural")].fine_f1.iloc[0]) -
                  float(_teach[_teach.architecture == a].fine_macro_f1.iloc[0])) <= 0.03) for a in ["shallow", "deep"]}
verdict = {"arm": "S_audit_followups", "S1_mde_le_0.03_both_cells": bool(S1),
           "S2": {a: {"recal_awbir": {k: analysis[a]["recal_awbir"][k] for k in ["p_method", "p_seed", "means", "mde", "max_mde", "n_holm_significant", "max_abs_mean", "rank_rho", "rei"]},
                      "recal_hsr_balanced": {k: analysis[a]["recal_hsr_balanced"][k] for k in ["p_method", "means", "n_holm_significant", "max_abs_mean"]},
                      "recal_b2a": {k: analysis[a]["recal_b2a"][k] for k in ["p_method", "means", "n_holm_significant", "max_abs_mean"]},
                      "five_seed_subset": analysis[a]["five_seed_subset_recal_awbir"], "collapses": analysis[a]["collapses"]} for a in analysis},
           "S3_reading": S3, "S3_random40": {"balanced_val_b2a_eval": float(fro[(fro.method == "random") & np.isclose(fro.target_flops, 0.40) & (fro.prior == "balanced")].eval_b2a.iloc[0]),
                                            "natural_val_b2a_eval": float(r40n.eval_b2a), "natural_val_b2a_batch": float(r40n.batch_b2a), "natural_val_b2a_recal": float(r40n.recal_b2a), "test_b2a_locked": 0.118,
                                            "benign_breakdown": ben.to_dict("records")},
           "S4_teacher_val_natural_within_0.03_of_test": S4,
           "natural_vs_balanced_validation_students": nat[nat.model != "dense"].groupby(["architecture", "prior"])[["fine_f1", "b2a", "awbir", "ece"]].mean().round(4).reset_index().to_dict("records"),
           "prereg": json.load(open(OUT / "S_PREREGISTRATION.json"))}
(OUT / "S_verdict.json").write_text(json.dumps(verdict, indent=2, default=lambda o: float(o)))
print(json.dumps({k: v for k, v in verdict.items() if k not in ("prereg", "S2", "natural_vs_balanced_validation_students")}, indent=2))
for a in analysis:
    print(a, "recal AWBIR:", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in verdict["S2"][a]["recal_awbir"].items() if k != "means"})

runs = pd.read_csv(OUT / "twenty_seed_runs.csv")
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, arch in zip(axes, ["shallow", "deep"]):
    sub = runs[runs.architecture == arch]
    data = [sub[sub.method == m].recal_awbir.values for m in METHODS]
    ax.boxplot(data, showmeans=True); ax.set_xticks(range(1, len(METHODS) + 1)); ax.set_xticklabels(METHODS)
    ax.set_title(f"{arch}, minimal recovery, 20 seeds, recalibrated"); ax.set_ylabel("AWBIR")
fig.tight_layout(); fig.savefig(OUT / "S_twenty_seeds.png", dpi=200); plt.show()

fig, ax = plt.subplots(figsize=(7, 3.4))
fs = fro[fro.prior == "natural"]; xs = np.arange(len(fs))
ax.bar(xs - 0.2, fs.eval_b2a, 0.4, label="stored statistics"); ax.bar(xs + 0.2, fs.recal_b2a, 0.4, label="recalibrated")
ax.set_xticks(xs); ax.set_xticklabels([f"{m}/{t:.2f}" for m, t in zip(fs.method, fs.target_flops)], rotation=45, ha="right", fontsize=7)
ax.set_yscale("symlog", linthresh=1e-3); ax.set_ylabel("benign escalation, natural-prior validation"); ax.legend(fontsize=7); ax.axhline(0.10, color="0.4", ls=":", lw=0.9)
fig.tight_layout(); fig.savefig(OUT / "S_frozen_natural_prior.png", dpi=200); plt.show(); print("written ->", OUT)
